In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

In [2]:
train = pd.read_csv("/home/akel/PycharmProjects/Kaggle/Diabetes_Prediction_Challenge/data/raw/train.csv")
test = pd.read_csv("/home/akel/PycharmProjects/Kaggle/Diabetes_Prediction_Challenge/data/raw/test.csv")
sample = pd.read_csv("/home/akel/PycharmProjects/Kaggle/Diabetes_Prediction_Challenge/data/raw/sample_submission.csv")

TARGET = train.columns[-1]
ID_COL = train.columns[0]

X = train.drop([TARGET], axis=1)
y = train[TARGET]
X_test = test.copy()


test_ids = test[ID_COL]

full = pd.concat([X, test], axis=0).reset_index(drop=True)


In [3]:
if 'BMI' in full.columns and 'Age' in full.columns:
    full["BMI_Age"] = full["BMI"] * full["Age"]

if 'Glucose' in full.columns and 'Insulin' in full.columns:
    full["Glucose_Insulin"] = full["Glucose"] * full["Insulin"]


for col in full.columns:
    if full[col].dtype == "object":
        full[col] = full[col].astype("category").cat.codes

# Split back
X = full.iloc[:len(train)]
X_test = full.iloc[len(train):]

In [4]:
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

In [5]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X))
preds = np.zeros(len(X_test))

In [8]:
models = [
    XGBClassifier(
        n_estimators=1200,
        learning_rate=0.02,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1,
        reg_lambda=2,
        eval_metric="logloss",
        random_state=42
    ),

    LGBMClassifier(
        n_estimators=1200,
        learning_rate=0.02,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ),

    CatBoostClassifier(
        iterations=1200,
        learning_rate=0.02,
        depth=5,
        verbose=0,
        random_state=42
    )
]

In [9]:
for model in models:
    print(f"\nTraining {model.__class__.__name__}")
    
    fold_oof = np.zeros(len(X))
    fold_pred = np.zeros(len(X_test))
    
    for fold, (trn_idx, val_idx) in enumerate(folds.split(X, y)):
        
        X_train, X_valid = X[trn_idx], X[val_idx]
        y_train, y_valid = y.iloc[trn_idx], y.iloc[val_idx]
        
        # SMOTE inside fold
        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        
        model.fit(X_train, y_train)
        
        fold_oof[val_idx] = model.predict_proba(X_valid)[:,1]
        fold_pred += model.predict_proba(X_test)[:,1] / folds.n_splits
        
        print("Fold", fold+1,
              "AUC:", roc_auc_score(y_valid, fold_oof[val_idx]))
    
    print("Full AUC:", roc_auc_score(y, fold_oof))
    
    oof += fold_oof / len(models)
    preds += fold_pred / len(models)


print("\nFINAL CV AUC:", roc_auc_score(y, oof))


Training XGBClassifier
Fold 1 AUC: 0.7147502508242376
Fold 2 AUC: 0.7139408074094963
Fold 3 AUC: 0.7147719105207716
Fold 4 AUC: 0.7150645385222892
Fold 5 AUC: 0.7146744352147685
Full AUC: 0.7146370416855689

Training LGBMClassifier
[LightGBM] [Info] Number of positive: 349045, number of negative: 349045
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.079887 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5680
[LightGBM] [Info] Number of data points in the train set: 698090, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Fold 1 AUC: 0.7252391093688313
[LightGBM] [Info] Number of positive: 349045, number of negative: 349045
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.055409 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5678
[LightGBM] [Info] Number 